In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
import os

api_key = os.getenv("GOOGLE_API_KEY")

if os.environ['GOOGLE_API_KEY']:
    print("Google api key is set")
else:
    raise ValueError("Gemini API key is not set")


llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", google_api_key=api_key, temperature=0.7)

Google api key is set


In [ ]:
from typing import TypedDict, List
from langchain_core.documents import Document
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    question: str
    path: str = Field("path for the documents")

class graph_schema(TypedDict):
    query: str
    question: str
    documents: List[Document]
    result: str
    path: str

llm_with_schema = llm.with_structured_output(llm_schema)
result = llm_with_schema.invoke("What is INVOLVE an AI powered on demand local service platform? examine a pdf named 29_Paper.pdf inside pdf folder of docs that is inside of C drive then Faheem then My_Programs Rag_Beginners")
result

c:\FAHEEM\My_Programs\RAG_Beginners\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


llm_schema(question='What is INVOLVE an AI powered on demand local service platform?', path='C:\\Faheem\\My_Programs\\Rag_Beginners\\docs\\pdf\\29_Paper.pdf')

In [ ]:
from RagIntegration import RagRetriever
from langchain_core.prompts import ChatPromptTemplate

retrieve = RagRetriever()

def retriever_node(state: graph_schema) -> graph_schema:
    query = state['query']
    response = llm_with_schema.invoke(query)
    state['question'] = response.question
    state['path'] = response.path
    retrieve.file_loader(state['path'])
    retrieve.text_splitting()
    retrieve.model_calling()
    result = retrieve.vector_embedding()

    return {
        "documents": result
    }
    
def generator_node(state: graph_schema) -> graph_schema:
    question = state['question']
    document = state['documents']

    context = '\n\n'.join(docs.page_content for docs in document)

    prompt = ChatPromptTemplate.from_messages([
        ("system", """
            You are a helpful assistant.
            Use only the information provided in the context and answer the question.
            If answer is not present in the context just say exactly.
            'I don't know'
            Context: {context}
        """),
        ("user", """Question: {question}""")
    ])

    chain = prompt | llm

    result = chain.invoke({"context": context, "question": question})

    return {
        "result": result.content
    }